# Marigold Art & Design Collective: Studio Space & Instructor Scheduling Analysis

This notebook prepares a room occupancy dataset for Tableau analysis focused on studio utilization, fill rate, ghost hours, and revenue efficiency.

In [1]:
import pandas as pd
import numpy as np

In [2]:
from google.colab import files
uploaded = files.upload()

Saving room occupancy.csv to room occupancy.csv


In [3]:
df = pd.read_csv("room occupancy.csv")
df.head()

,Date,Time,S1_Temp,S2_Temp,S3_Temp,S4_Temp,S1_Light,S2_Light,S3_Light,S4_Light,S1_Sound,S2_Sound,S3_Sound,S4_Sound,S5_CO2,S5_CO2_Slope,S6_PIR,S7_PIR,Room_Occupancy_Count
0,22-12-2017,10:49:41,24.94,24.75,24.56,25.38,121,34,53,40,0.08,0.19,0.06,0.06,390,0.769231,0,0,1
1,22-12-2017,10:50:12,24.94,24.75,24.56,25.44,121,33,53,40,0.93,0.05,0.06,0.06,390,0.646154,0,0,1
2,22-12-2017,10:50:42,25.00,24.75,24.50,25.44,121,34,53,40,0.43,0.11,0.08,0.06,390,0.519231,0,0,1
3,22-12-2017,10:51:13,25.00,24.75,24.56,25.44,121,34,53,40,0.41,0.10,0.10,0.09,390,0.388462,0,0,1
4,22-12-2017,10:51:44,25.00,24.75,24.56,25.44,121,34,54,40,0.18,0.06,0.06,0.06,390,0.253846,0,0,1


## Check the dataset shape and columns

In [4]:
print(df.shape)
print(df.columns.tolist())

(10129, 19)
['Date', 'Time', 'S1_Temp', 'S2_Temp', 'S3_Temp', 'S4_Temp', 'S1_Light', 'S2_Light', 'S3_Light', 'S4_Light', 'S1_Sound', 'S2_Sound', 'S3_Sound', 'S4_Sound', 'S5_CO2', 'S5_CO2_Slope', 'S6_PIR', 'S7_PIR', 'Room_Occupancy_Count']


## Clean the column names

In [5]:
df.columns = (
    df.columns
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

for col in df.columns:
    print(repr(col))

'Date'
'Time'
'S1_Temp'
'S2_Temp'
'S3_Temp'
'S4_Temp'
'S1_Light'
'S2_Light'
'S3_Light'
'S4_Light'
'S1_Sound'
'S2_Sound'
'S3_Sound'
'S4_Sound'
'S5_CO2'
'S5_CO2_Slope'
'S6_PIR'
'S7_PIR'
'Room_Occupancy_Count'


## Check missing values

In [6]:
print(df.isnull().sum())

Date                    0
Time                    0
S1_Temp                 0
S2_Temp                 0
S3_Temp                 0
S4_Temp                 0
S1_Light                0
S2_Light                0
S3_Light                0
S4_Light                0
S1_Sound                0
S2_Sound                0
S3_Sound                0
S4_Sound                0
S5_CO2                  0
S5_CO2_Slope            0
S6_PIR                  0
S7_PIR                  0
Room_Occupancy_Count    0
dtype: int64


## Preview the main fields

In [7]:
df.head(15)

,Date,Time,S1_Temp,S2_Temp,S3_Temp,S4_Temp,S1_Light,S2_Light,S3_Light,S4_Light,S1_Sound,S2_Sound,S3_Sound,S4_Sound,S5_CO2,S5_CO2_Slope,S6_PIR,S7_PIR,Room_Occupancy_Count
0,22-12-2017,10:49:41,24.94,24.75,24.56,25.38,121,34,53,40,0.08,0.19,0.06,0.06,390,0.769231,0,0,1
1,22-12-2017,10:50:12,24.94,24.75,24.56,25.44,121,33,53,40,0.93,0.05,0.06,0.06,390,0.646154,0,0,1
2,22-12-2017,10:50:42,25.00,24.75,24.50,25.44,121,34,53,40,0.43,0.11,0.08,0.06,390,0.519231,0,0,1
3,22-12-2017,10:51:13,25.00,24.75,24.56,25.44,121,34,53,40,0.41,0.10,0.10,0.09,390,0.388462,0,0,1
4,22-12-2017,10:51:44,25.00,24.75,24.56,25.44,121,34,54,40,0.18,0.06,0.06,0.06,390,0.253846,0,0,1
5,22-12-2017,10:52:14,25.00,24.81,24.56,25.44,121,34,54,40,0.13,0.06,0.06,0.07,390,0.165385,0,0,1
6,22-12-2017,10:52:45,25.00,24.75,24.56,25.44,120,34,54,40,1.39,0.32,0.43,0.06,390,0.076923,1,0,1
7,22-12-2017,10:53:15,25.00,24.81,24.56,25.44,121,34,54,41,0.09,0.06,0.09,0.05,390,-0.011538,0,0,1
8,22-12-2017,10:53:46,25.00,24.81,24.56,25.50,122,35,56,43,0.09,0.05,0.06,0.13,390,-0.100000,0,0,1
9,22-12-2017,10:54:17,25.00,24.81,24.56,25.50,101,34,57,43,3.84,0.64,0.48,0.39,390,-0.188462,1,1,1


## Check numeric columns and basic summary

In [8]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
print(numeric_cols)
df[numeric_cols].describe()

['S1_Temp', 'S2_Temp', 'S3_Temp', 'S4_Temp', 'S1_Light', 'S2_Light', 'S3_Light', 'S4_Light', 'S1_Sound', 'S2_Sound', 'S3_Sound', 'S4_Sound', 'S5_CO2', 'S5_CO2_Slope', 'S6_PIR', 'S7_PIR', 'Room_Occupancy_Count']


,S1_Temp,S2_Temp,S3_Temp,S4_Temp,S1_Light,S2_Light,S3_Light,S4_Light,S1_Sound,S2_Sound,S3_Sound,S4_Sound,S5_CO2,S5_CO2_Slope,S6_PIR,S7_PIR,Room_Occupancy_Count
count,10129.000000,10129.000000,10129.000000,10129.000000,10129.000000,10129.00000,10129.000000,10129.000000,10129.000000,10129.000000,10129.000000,10129.000000,10129.000000,10129.000000,10129.000000,10129.000000,10129.000000
mean,25.454012,25.546059,25.056621,25.754125,25.445059,26.01629,34.248494,13.220259,0.168178,0.120066,0.158119,0.103840,460.860401,-0.004830,0.090137,0.079574,0.398559
std,0.351351,0.586325,0.427283,0.356434,51.011264,67.30417,58.400744,19.602219,0.316709,0.266503,0.413637,0.120683,199.964940,1.164990,0.286392,0.270645,0.893633
min,24.940000,24.750000,24.440000,24.940000,0.000000,0.00000,0.000000,0.000000,0.060000,0.040000,0.040000,0.050000,345.000000,-6.296154,0.000000,0.000000,0.000000
25%,25.190000,25.190000,24.690000,25.440000,0.000000,0.00000,0.000000,0.000000,0.070000,0.050000,0.060000,0.060000,355.000000,-0.046154,0.000000,0.000000,0.000000
50%,25.380000,25.380000,24.940000,25.750000,0.000000,0.00000,0.000000,0.000000,0.080000,0.050000,0.060000,0.080000,360.000000,0.000000,0.000000,0.000000,0.000000
75%,25.630000,25.630000,25.380000,26.000000,12.000000,14.00000,50.000000,22.000000,0.080000,0.060000,0.070000,0.100000,465.000000,0.000000,0.000000,0.000000,0.000000
max,26.380000,29.000000,26.190000,26.560000,165.000000,258.00000,280.000000,74.000000,3.880000,3.440000,3.670000,3.400000,1270.000000,8.980769,1.000000,1.000000,3.000000


## Check the key occupancy fields directly

In [9]:
df[[
    "Date",
    "Time",
    "Room_Occupancy_Count"
]].head(20)

,Date,Time,Room_Occupancy_Count
0,22-12-2017,10:49:41,1
1,22-12-2017,10:50:12,1
2,22-12-2017,10:50:42,1
3,22-12-2017,10:51:13,1
4,22-12-2017,10:51:44,1
5,22-12-2017,10:52:14,1
6,22-12-2017,10:52:45,1
7,22-12-2017,10:53:15,1
8,22-12-2017,10:53:46,1
9,22-12-2017,10:54:17,1


## Check the date and time variety

This helps confirm whether the dataset is strong enough for hour-based and weekday-based analysis.

In [10]:
print("Unique dates:", df["Date"].nunique())
print("Unique times:", df["Time"].nunique())
print("Min occupancy:", df["Room_Occupancy_Count"].min())
print("Max occupancy:", df["Room_Occupancy_Count"].max())

Unique dates: 7
Unique times: 10129
Min occupancy: 0
Max occupancy: 3


# Step 2: Clean the occupancy data and create the core operational fields

In this step, we convert the date and time fields, make sure occupancy values are usable, and create the first operational fields needed for studio utilization analysis.

## Keep only the columns needed from the source file

In [11]:
columns_needed = [
    "Date",
    "Time",
    "Room_Occupancy_Count"
]

df = df[columns_needed].copy()
df.head()

,Date,Time,Room_Occupancy_Count
0,22-12-2017,10:49:41,1
1,22-12-2017,10:50:12,1
2,22-12-2017,10:50:42,1
3,22-12-2017,10:51:13,1
4,22-12-2017,10:51:44,1


## Remove rows with missing values in the core fields

In [12]:
df = df.dropna(subset=["Date", "Time", "Room_Occupancy_Count"]).copy()
print(df.shape)

(10129, 3)


## Convert the occupancy field to numeric

In [13]:
df["Room_Occupancy_Count"] = pd.to_numeric(df["Room_Occupancy_Count"], errors="coerce")
print(df["Room_Occupancy_Count"].dtype)

int64


## Remove rows that became invalid after numeric conversion

In [14]:
df = df.dropna(subset=["Room_Occupancy_Count"]).copy()
print(df.shape)

(10129, 3)


## Create a combined timestamp field

In [15]:
df["Timestamp"] = pd.to_datetime(
    df["Date"].astype(str) + " " + df["Time"].astype(str),
    errors="coerce"
)

print(df["Timestamp"].dtype)
df.head()

datetime64[ns]


/tmp/ipykernel_19568/2373391556.py:1: UserWarning: Parsing dates in %d-%m-%Y %H:%M:%S format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["Timestamp"] = pd.to_datetime(


,Date,Time,Room_Occupancy_Count,Timestamp
0,22-12-2017,10:49:41,1,2017-12-22 10:49:41
1,22-12-2017,10:50:12,1,2017-12-22 10:50:12
2,22-12-2017,10:50:42,1,2017-12-22 10:50:42
3,22-12-2017,10:51:13,1,2017-12-22 10:51:13
4,22-12-2017,10:51:44,1,2017-12-22 10:51:44


## Remove rows with invalid timestamps

In [16]:
df = df.dropna(subset=["Timestamp"]).copy()
print(df.shape)

(10129, 4)


## Create hour and weekday fields

These will support peak vs off-peak and weekday analysis in Tableau.

In [17]:
df["Hour"] = df["Timestamp"].dt.hour
df["Weekday"] = df["Timestamp"].dt.day_name()
df["Month"] = df["Timestamp"].dt.month_name()

df[["Timestamp", "Hour", "Weekday", "Month"]].head()

,Timestamp,Hour,Weekday,Month
0,2017-12-22 10:49:41,10,Friday,December
1,2017-12-22 10:50:12,10,Friday,December
2,2017-12-22 10:50:42,10,Friday,December
3,2017-12-22 10:51:13,10,Friday,December
4,2017-12-22 10:51:44,10,Friday,December


## Create a peak vs off-peak field

This makes it easier to compare high-demand periods to quieter periods.

In [18]:
def map_peak_period(hour):
    if hour in [10, 11, 12, 13, 17, 18, 19]:
        return "Peak"
    else:
        return "Off-Peak"

df["Peak_Period"] = df["Hour"].apply(map_peak_period)
df[["Hour", "Peak_Period"]].drop_duplicates().sort_values("Hour")

,Hour,Peak_Period
1462,0,Off-Peak
1577,1,Off-Peak
1693,2,Off-Peak
1809,3,Off-Peak
1925,4,Off-Peak
2042,5,Off-Peak
2158,6,Off-Peak
2274,7,Off-Peak
2388,8,Off-Peak
2504,9,Off-Peak


## Preview the cleaned occupancy dataset

In [19]:
print(df.shape)
df.head()

(10129, 8)


,Date,Time,Room_Occupancy_Count,Timestamp,Hour,Weekday,Month,Peak_Period
0,22-12-2017,10:49:41,1,2017-12-22 10:49:41,10,Friday,December,Peak
1,22-12-2017,10:50:12,1,2017-12-22 10:50:12,10,Friday,December,Peak
2,22-12-2017,10:50:42,1,2017-12-22 10:50:42,10,Friday,December,Peak
3,22-12-2017,10:51:13,1,2017-12-22 10:51:13,10,Friday,December,Peak
4,22-12-2017,10:51:44,1,2017-12-22 10:51:44,10,Friday,December,Peak


# Step 3: Create studio, capacity, fill-rate, and revenue fields

In this step, we create the studio-style operational fields needed for Tableau analysis, including room identity, class medium, room capacity, fill rate, ghost hours, revenue, and instructor tenure.

## Create a simulated studio name field

Since the source file does not include room names, we assign a realistic set of studio spaces.

In [20]:
studios = [
    "Sunlit Pottery Studio",
    "North Loft Painting Studio",
    "Digital Lab East",
    "Garden Ceramics Room"
]

df["Studio_Name"] = [studios[i % len(studios)] for i in range(len(df))]
df["Studio_Name"].value_counts()

,count
Studio_Name,
Sunlit Pottery Studio,2533
North Loft Painting Studio,2532
Digital Lab East,2532
Garden Ceramics Room,2532


## Create a class medium field

This maps each studio into an art-medium category for the dashboard toggle and comparisons.

In [21]:
medium_map = {
    "Sunlit Pottery Studio": "Pottery",
    "North Loft Painting Studio": "Painting",
    "Digital Lab East": "Digital",
    "Garden Ceramics Room": "Pottery"
}

df["Class_Medium"] = df["Studio_Name"].map(medium_map)
df[["Studio_Name", "Class_Medium"]].drop_duplicates()

,Studio_Name,Class_Medium
0,Sunlit Pottery Studio,Pottery
1,North Loft Painting Studio,Painting
2,Digital Lab East,Digital
3,Garden Ceramics Room,Pottery


## Create room capacity and floor area fields

These give us the base for fill-rate and revenue-per-square-foot analysis.

In [22]:
capacity_map = {
    "Sunlit Pottery Studio": 18,
    "North Loft Painting Studio": 22,
    "Digital Lab East": 16,
    "Garden Ceramics Room": 20
}

area_map = {
    "Sunlit Pottery Studio": 650,
    "North Loft Painting Studio": 780,
    "Digital Lab East": 520,
    "Garden Ceramics Room": 700
}

df["Room_Capacity"] = df["Studio_Name"].map(capacity_map)
df["Floor_Area_SqFt"] = df["Studio_Name"].map(area_map)

df[["Studio_Name", "Room_Capacity", "Floor_Area_SqFt"]].drop_duplicates()

,Studio_Name,Room_Capacity,Floor_Area_SqFt
0,Sunlit Pottery Studio,18,650
1,North Loft Painting Studio,22,780
2,Digital Lab East,16,520
3,Garden Ceramics Room,20,700


## Create fill rate

This measures how full the studio is relative to capacity.

In [23]:
df["Fill_Rate"] = (df["Room_Occupancy_Count"] / df["Room_Capacity"]).clip(lower=0, upper=1).round(3)
df[["Room_Occupancy_Count", "Room_Capacity", "Fill_Rate"]].head()

,Room_Occupancy_Count,Room_Capacity,Fill_Rate
0,1,18,0.056
1,1,22,0.045
2,1,16,0.062
3,1,20,0.050
4,1,18,0.056


## Create a ghost hour flag

Ghost hours are low-utilization periods when the studio is underused.

In [24]:
df["Ghost_Hour_Flag"] = np.where(df["Fill_Rate"] < 0.25, "Ghost Hour", "Active Hour")
df["Ghost_Hour_Flag"].value_counts()

,count
Ghost_Hour_Flag,
Ghost Hour,10129


## Create a simulated revenue field

We assign a realistic revenue-per-attendee by class medium, then calculate session revenue.

In [25]:
revenue_per_attendee_map = {
    "Pottery": 38,
    "Painting": 32,
    "Digital": 45
}

df["Revenue_per_Attendee"] = df["Class_Medium"].map(revenue_per_attendee_map)
df["Revenue"] = (df["Room_Occupancy_Count"] * df["Revenue_per_Attendee"]).round(2)

df[["Class_Medium", "Room_Occupancy_Count", "Revenue_per_Attendee", "Revenue"]].head()

,Class_Medium,Room_Occupancy_Count,Revenue_per_Attendee,Revenue
0,Pottery,1,38,38
1,Painting,1,32,32
2,Digital,1,45,45
3,Pottery,1,38,38
4,Pottery,1,38,38


## Create revenue per square foot

This supports studio efficiency analysis.

In [26]:
df["Revenue_per_SqFt"] = (df["Revenue"] / df["Floor_Area_SqFt"]).round(3)
df[["Revenue", "Floor_Area_SqFt", "Revenue_per_SqFt"]].head()

,Revenue,Floor_Area_SqFt,Revenue_per_SqFt
0,38,650,0.058
1,32,780,0.041
2,45,520,0.087
3,38,700,0.054
4,38,650,0.058


## Create a simulated instructor tenure band

This gives us a clean instructor-experience dimension for the dashboard.

In [27]:
tenure_map = {
    "Sunlit Pottery Studio": "Senior Instructor",
    "North Loft Painting Studio": "Mid-Level Instructor",
    "Digital Lab East": "Senior Instructor",
    "Garden Ceramics Room": "New Instructor"
}

df["Instructor_Tenure_Band"] = df["Studio_Name"].map(tenure_map)
df[["Studio_Name", "Instructor_Tenure_Band"]].drop_duplicates()

,Studio_Name,Instructor_Tenure_Band
0,Sunlit Pottery Studio,Senior Instructor
1,North Loft Painting Studio,Mid-Level Instructor
2,Digital Lab East,Senior Instructor
3,Garden Ceramics Room,New Instructor


## Preview the enhanced studio dataset

In [28]:
print(df.shape)
df[[
    "Timestamp",
    "Studio_Name",
    "Class_Medium",
    "Hour",
    "Weekday",
    "Room_Occupancy_Count",
    "Room_Capacity",
    "Fill_Rate",
    "Ghost_Hour_Flag",
    "Revenue",
    "Revenue_per_SqFt",
    "Instructor_Tenure_Band"
]].head(20)

(10129, 18)


,Timestamp,Studio_Name,Class_Medium,Hour,Weekday,Room_Occupancy_Count,Room_Capacity,Fill_Rate,Ghost_Hour_Flag,Revenue,Revenue_per_SqFt,Instructor_Tenure_Band
0,2017-12-22 10:49:41,Sunlit Pottery Studio,Pottery,10,Friday,1,18,0.056,Ghost Hour,38,0.058,Senior Instructor
1,2017-12-22 10:50:12,North Loft Painting Studio,Painting,10,Friday,1,22,0.045,Ghost Hour,32,0.041,Mid-Level Instructor
2,2017-12-22 10:50:42,Digital Lab East,Digital,10,Friday,1,16,0.062,Ghost Hour,45,0.087,Senior Instructor
3,2017-12-22 10:51:13,Garden Ceramics Room,Pottery,10,Friday,1,20,0.050,Ghost Hour,38,0.054,New Instructor
4,2017-12-22 10:51:44,Sunlit Pottery Studio,Pottery,10,Friday,1,18,0.056,Ghost Hour,38,0.058,Senior Instructor
5,2017-12-22 10:52:14,North Loft Painting Studio,Painting,10,Friday,1,22,0.045,Ghost Hour,32,0.041,Mid-Level Instructor
6,2017-12-22 10:52:45,Digital Lab East,Digital,10,Friday,1,16,0.062,Ghost Hour,45,0.087,Senior Instructor
7,2017-12-22 10:53:15,Garden Ceramics Room,Pottery,10,Friday,1,20,0.050,Ghost Hour,38,0.054,New Instructor
8,2017-12-22 10:53:46,Sunlit Pottery Studio,Pottery,10,Friday,1,18,0.056,Ghost Hour,38,0.058,Senior Instructor
9,2017-12-22 10:54:17,North Loft Painting Studio,Painting,10,Friday,1,22,0.045,Ghost Hour,32,0.041,Mid-Level Instructor


## Save the final Tableau-ready dataset

In [29]:
df.to_csv("marigold_studio_ready.csv", index=False)
print("Saved as marigold_studio_ready.csv")

Saved as marigold_studio_ready.csv


## Download the final dataset in Google Colab

In [30]:
from google.colab import files
files.download("marigold_studio_ready.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>